# Chapter 1 — PyTorch Warmup: Tensor, Device, Autograd

This notebook is the runnable companion to `docs/01_tensor_autograd_pytorch.md`.

Every cell below is a *minimum-viable* demonstration of one concept. Run them
in order. The notebook is generated by `src/build_chapter_01_tensor_autograd.py`
— if you want to change something, edit the builder, not this `.ipynb`.

**Sections.** Tensor basics → creation → reshaping → broadcasting → CPU/GPU →
autograd → hand-derivative cross-check → `no_grad` & `detach`.

## Setup

Import PyTorch and confirm the version + device.

In [1]:
import torch
import numpy as np

print("torch", torch.__version__)
print("cuda available:", torch.cuda.is_available())

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

torch 2.12.0+cu130
cuda available: False
device: cpu


/home/bangbc/miniforge3/envs/aicourse/lib/python3.11/site-packages/torch/cuda/__init__.py:187: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12040). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


## 1. The tensor — shape, dtype, device

Read off these three attributes for every tensor you ever see.

In [2]:
x = torch.tensor([[1.0, 2.0, 3.0],
                  [4.0, 5.0, 6.0]])

print("x =\n", x)
print("shape:", x.shape)
print("dtype:", x.dtype)
print("device:", x.device)

x =
 tensor([[1., 2., 3.],
        [4., 5., 6.]])
shape: torch.Size([2, 3])
dtype: torch.float32
device: cpu


### Tensor of different ranks

The course will keep showing you four shape patterns:

- `(B,)` — batch of scalars (e.g. class labels).
- `(B, D)` — batch of D-dim vectors (MLP input).
- `(B, C, H, W)` — batch of images (CNN input).
- `(B, T, D)` — batch of sequences of length T (RNN, Transformer).

Build one of each and read off the shape.

In [3]:
labels  = torch.randint(0, 10, (32,))          # (B,)         class indices
vectors = torch.randn(32, 128)                  # (B, D)       MLP-style
images  = torch.randn(32, 3, 224, 224)          # (B, C, H, W) image batch
seqs    = torch.randn(32, 50, 128)              # (B, T, D)    sequence batch

for name, t in [("labels", labels), ("vectors", vectors),
                ("images", images), ("seqs", seqs)]:
    print(f"{name:8s} shape={tuple(t.shape)} dtype={t.dtype}")

labels   shape=(32,) dtype=torch.int64
vectors  shape=(32, 128) dtype=torch.float32
images   shape=(32, 3, 224, 224) dtype=torch.float32
seqs     shape=(32, 50, 128) dtype=torch.float32


## 2. Common ways to create a tensor

Memorize these. You will type them out hundreds of times.

In [4]:
print("zeros:\n",   torch.zeros(2, 3))
print("ones:\n",    torch.ones(2, 3))
print("eye:\n",     torch.eye(3))
print("arange:\n",  torch.arange(0, 10, 2))
print("linspace:\n",torch.linspace(0, 1, 5))
print("randn:\n",   torch.randn(2, 3))
print("randint:\n", torch.randint(0, 10, (2, 3)))

# From a Python list and a NumPy array.
print("from list:\n",  torch.tensor([[1, 2], [3, 4]]))
print("from numpy:\n", torch.from_numpy(np.arange(6).reshape(2, 3)))

zeros:
 tensor([[0., 0., 0.],
        [0., 0., 0.]])
ones:
 tensor([[1., 1., 1.],
        [1., 1., 1.]])
eye:
 tensor([[1., 0., 0.],
        [0., 1., 0.],
        [0., 0., 1.]])
arange:
 tensor([0, 2, 4, 6, 8])
linspace:
 tensor([0.0000, 0.2500, 0.5000, 0.7500, 1.0000])
randn:
 tensor([[ 0.0872, -1.4335,  0.7419],
        [-1.3123, -0.0250,  0.5016]])
randint:
 tensor([[1, 3, 1],
        [0, 1, 7]])
from list:
 tensor([[1, 2],
        [3, 4]])
from numpy:
 tensor([[0, 1, 2],
        [3, 4, 5]])


## 3. Reshaping — `view`, `reshape`, `permute`, `squeeze`, `unsqueeze`

`view` is cheap (shares storage) but needs contiguous memory; `reshape` is
flexible (may copy). After a `permute`, the tensor is non-contiguous — use
`reshape` or `.contiguous().view(...)`.

In [5]:
x = torch.arange(24).reshape(2, 3, 4)
print("original shape:", x.shape)

# view: same data, no copy — needs contiguous memory.
print("view(2, 12):", x.view(2, 12).shape)

# reshape: works on non-contiguous tensors too.
print("reshape(2, 12):", x.reshape(2, 12).shape)

# permute: reorder axes.
xp = x.permute(0, 2, 1)
print("permute(0,2,1):", xp.shape, "contiguous?", xp.is_contiguous())

# squeeze / unsqueeze: drop / add a size-1 dim.
y = torch.zeros(1, 3, 4)
print("squeeze(0):",   y.squeeze(0).shape)
print("unsqueeze(0):", x.unsqueeze(0).shape)

# flatten from a given dim onward.
print("flatten(1):", x.flatten(start_dim=1).shape)

original shape: torch.Size([2, 3, 4])
view(2, 12): torch.Size([2, 12])
reshape(2, 12): torch.Size([2, 12])
permute(0,2,1): torch.Size([2, 4, 3]) contiguous? False
squeeze(0): torch.Size([3, 4])
unsqueeze(0): torch.Size([1, 2, 3, 4])
flatten(1): torch.Size([2, 12])


## 4. Broadcasting

Two shapes are compatible if, reading from the right, every pair is either
equal or one of them is `1`. Use broadcasting instead of tiling.

In [6]:
a = torch.randn(3, 1)
b = torch.randn(1, 4)
c = a + b
print("a:", a.shape, "+ b:", b.shape, "->", c.shape)

# The most common pattern: per-feature bias broadcast over a batch.
x    = torch.randn(32, 128)
bias = torch.randn(128)
y    = x + bias
print("x:", x.shape, "+ bias:", bias.shape, "->", y.shape)

a: torch.Size([3, 1]) + b: torch.Size([1, 4]) -> torch.Size([3, 4])
x: torch.Size([32, 128]) + bias: torch.Size([128]) -> torch.Size([32, 128])


## 5. CPU vs. GPU — `.to(device)`

Decide `DEVICE` once at the top of the script (we already did, in the Setup
cell). Move both data and model to that device *before* the training loop.

In [7]:
x_cpu = torch.randn(4, 4)
x_dev = x_cpu.to(DEVICE)
print("on cpu :", x_cpu.device)
print("on dev :", x_dev.device)

# Bring a tensor back to CPU when you need NumPy or matplotlib.
arr = x_dev.cpu().numpy()
print("back to numpy:", type(arr).__name__, arr.shape)

on cpu : cpu
on dev : cpu
back to numpy: ndarray (4, 4)


## 6. Automatic differentiation — `requires_grad`, `.backward()`, `.grad`

A scalar loss + `.backward()` populates `.grad` on every tensor that
`requires_grad`. Below we compute `y = x^2 + 2x + 1` at `x = 3`. The
hand-derived derivative is `dy/dx = 2x + 2 = 8`.

In [8]:
x = torch.tensor([3.0], requires_grad=True)
y = x ** 2 + 2 * x + 1
y.backward()

print("y =", y.item(), "(should be 16.0)")
print("x.grad =", x.grad.item(), "(should be 8.0)")

y = 16.0 (should be 16.0)
x.grad = 8.0 (should be 8.0)


### `y = x.sum()` — gradient is all ones

For `y = sum(x)`, `dy/dx_i = 1` for every i, so `x.grad` is a tensor of ones
the same shape as `x`.

In [9]:
x = torch.randn(2, 3, requires_grad=True)
y = x.sum()
y.backward()
print("x.grad:\n", x.grad)

x.grad:
 tensor([[1., 1., 1.],
        [1., 1., 1.]])


## 7. Hand-derivative cross-check — a 2-variable function

Function: `f(a, b) = a^2 * b + sin(b)`. Hand-derived:

- `df/da = 2 * a * b`
- `df/db = a^2 + cos(b)`

We compute both numerically (with autograd) and compare to the formula.

In [10]:
import math

a = torch.tensor([1.5], requires_grad=True)
b = torch.tensor([0.7], requires_grad=True)

f = a ** 2 * b + torch.sin(b)
f.backward()

a_grad_hand = 2 * a.item() * b.item()
b_grad_hand = a.item() ** 2 + math.cos(b.item())

print(f"df/da: autograd={a.grad.item():.6f}  hand={a_grad_hand:.6f}")
print(f"df/db: autograd={b.grad.item():.6f}  hand={b_grad_hand:.6f}")

assert torch.isclose(a.grad, torch.tensor([a_grad_hand]), atol=1e-5)
assert torch.isclose(b.grad, torch.tensor([b_grad_hand]), atol=1e-5)
print("OK — autograd matches hand-derived gradients.")

df/da: autograd=2.100000  hand=2.100000
df/db: autograd=3.014842  hand=3.014842
OK — autograd matches hand-derived gradients.


## 8. `torch.no_grad()` and `.detach()`

Wrap evaluation code in `torch.no_grad()` so PyTorch does not build a graph.
Use `.detach()` to take a tensor *out* of the graph (for logging, plotting).

In [11]:
x = torch.randn(4, requires_grad=True)

with torch.no_grad():
    y_eval = x * 2
    print("y_eval.requires_grad:", y_eval.requires_grad)

z = (x * 3).detach()
print("z.requires_grad:", z.requires_grad)

# A common pattern: log the loss value as a plain Python float.
loss = (x ** 2).sum()
loss_value = loss.detach().cpu().item()
print("loss value (python float):", loss_value)

y_eval.requires_grad: False
z.requires_grad: False
loss value (python float): 6.609248638153076


## Self-test

Predict the output of each line *before* running it, then run and verify.

If you get any wrong, re-read the corresponding section in
`docs/01_tensor_autograd_pytorch.md`.

In [12]:
# Q1: shape after permute
t = torch.randn(2, 3, 4)
print("Q1:", t.permute(0, 2, 1).shape)             # (2, 4, 3)

# Q2: gradient of sum
x = torch.ones(3, requires_grad=True)
x.sum().backward()
print("Q2:", x.grad)                                # tensor([1., 1., 1.])

# Q3: broadcasting result shape
a = torch.zeros(5, 1, 3)
b = torch.zeros(   4, 3)
print("Q3:", (a + b).shape)                         # (5, 4, 3)

# Q4: detach + cpu + item
y = torch.tensor([2.5], requires_grad=True)
print("Q4:", (y * 4).detach().cpu().item())         # 10.0

Q1: torch.Size([2, 4, 3])
Q2: tensor([1., 1., 1.])
Q3: torch.Size([5, 4, 3])
Q4: 10.0


## Next

- `docs/02_mlp_backpropagation.md` — logistic regression to MLP, then manual
  backprop with a NumPy cross-check.
- `notebooks/chapter_02_mlp_from_scratch.ipynb` — XOR + 2-2-1 MLP + the
  hand-vs-autograd gradient check.